**Anomalib**

Deep Learning Library which has SOTA anomaly detection algorithms

In [10]:
pip install anomalib


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
from anomalib.data import MVTecAD

categories = ["bottle","cable","capsule","carpet","grid","hazelnut","leather","metal_nut","pill","screw","tile","toothbrush","transistor","wood","zipper"]
datamodules = {}
train_loaders = {}

for cat in categories:
    datamodule = MVTecAD(
        root="./datasets/MVTecAD",
        category=cat,
        train_batch_size=32,
        eval_batch_size=32
    )
    
    datamodule.prepare_data()
    datamodule.setup()
    
    datamodules[cat] = datamodule
    train_loaders[cat] = datamodule.train_dataloader()

In [12]:
import os
import glob
import numpy as np
from PIL import Image
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torchvision import models,transforms
from torchvision.models import ResNet50_Weights

## Data Loading and Processing

In [13]:
class MVTecDataset(Dataset):
    def __init__(self,root_dir,category,is_train=True,transform=None,mask_transform=None):
        self.transform = transform
        self.mask_transform = mask_transform
        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        
        if is_train:
            img_dir = os.path.join(root_dir,category,"train","good")
            paths = glob.glob(os.path.join(img_dir,"*.png"))
            self.image_paths.extend(paths)
            self.labels.extend([0]*len(paths)) # 0 for no defect
            self.mask_paths.extend([None]*len(paths)) # No mask for non defective
        
        else:
            test_dir = os.path.join(root_dir,category,"test")
            for defect in os.listdir(test_dir):
                if defect.startswith('.'):
                    continue
                paths = glob.glob(os.path.join(test_dir,defect,"*.png"))
                self.image_paths.extend(paths)
                
                if defect == "good":
                    self.labels.extend([0]*len(paths))
                    self.mask_paths.extend([None]*len(paths))
                else:
                    self.labels.extend([1]*len(paths))
                    for p in paths:
                        mask_path = os.path.join(root_dir,category,"ground_truth",defect,os.path.basename(p).replace(".png","_mask.png"))
                        self.mask_paths.append(mask_path)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self,idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        mask_path = self.mask_paths[idx]
        
        if mask_path and os.path.exists(mask_path):
            mask = Image.open(mask_path).convert("L")
            if self.mask_transform:
                mask = self.mask_transform(mask)
        else:
            mask = torch.zeros((1,image.shape[1],image.shape[2]))
        return image,label,mask

## Feature Extraction from ResNet Layers

Features from Layer 2 and 3 of the ResNet are extracted : which are neither too coarse nor too fine to work with (hence did not choose Layer 1 or 4)


Layer 2 : 512 Channels

Layer 3 : 1024 Channels

But Layer 3 image is of smaller size than Layer 2, so up-scaled using billinear interpolation

In [ ]:
class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        
        # 512 Channels
        self.encoder_l2 = nn.Sequential(
            resnet.conv1,resnet.bn1,resnet.relu,resnet.maxpool,resnet.layer1,resnet.layer2
        )
        
        # 1024 Channels
        self.encoder_l3 = resnet.layer3
        
        for param in self.parameters():
            param.requires_grad = False
    
    def forward(self,x):
        with torch.no_grad():
            feat2 = self.encoder_l2(x) # Larger Image
            feat3 = self.encoder_l3(feat2) # Smaller Image : Needs to be resized before concat
            
            feat3_up = nn.functional.interpolate(feat3,size=feat2.shape[2:],mode="bilinear",align_corners=False)
            return torch.cat([feat2,feat3_up],dim=1)

## 1x1 Convolutional Autoencoder

1536 -> 512 -> 128 -> 512 -> 1536 network which needs to reconstruct features after compressing them

The 128 dimensional bottleneck before expanding back to 1536 against a MSE Loss forces network to learn the core essential patterns 

Trained only on train/good so that model learns to encode and decode perfect textures

When OOD image is passed, the bottleneck will fail to compress to 128 features and reconstruct from it, as a result MSE between input and reconstrcuted features will be very large

*Similar idea as 1_MSP_Baseline in OOD*

In [15]:
class ConvAutoEncoder(nn.Module):
    def __init__(self,in_channels=1536):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=in_channels,out_channels=512,kernel_size=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512,out_channels=128,kernel_size=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels=128,out_channels=512,kernel_size=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=512,out_channels=in_channels,kernel_size=1)
        )
    
    def forward(self,x):
        return self.decoder(self.encoder(x))

In [16]:
def train_evaluate(category="bottle",root_dir="./data/mvtec_ad",epochs=20,batch_size=32):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
    
    img_size = 256
    transform = transforms.Compose([
        transforms.Resize((img_size,img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    mask_transform = transforms.Compose([
        transforms.Resize((img_size,img_size)),
        transforms.ToTensor()
    ])
    
    train_loader = DataLoader(MVTecDataset(root_dir,category,is_train=True,transform=transform),batch_size=batch_size,shuffle=True)
    test_loader = DataLoader(MVTecDataset(root_dir,category,is_train=False,transform=transform,mask_transform=mask_transform),batch_size=1,shuffle=False)
    
    extractor = ResNetFeatureExtractor().to(device).eval()
    autoencoder = ConvAutoEncoder().to(device)
    optimizer = torch.optim.Adam(autoencoder.parameters(),lr=1e-3)
    criterion = nn.MSELoss()
    
    print(f"Autoencoder training for {category.capitalize()}")
    autoencoder.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for images,labels,masks in train_loader:
            images = images.to(device)
            features = extractor(images)
            
            optimizer.zero_grad()
            reconstr = autoencoder(features)
            loss = criterion(reconstr,features)
            loss.backward()
            optimizer.step()
            
            epoch_loss+=loss.item()
            
        print(f"Epoch {epoch+1}/{epochs}  |  MSE Loss : {epoch_loss/len(train_loader):.4f}")
    
    print(f"Evaluating {category.capitalize()}")
    autoencoder.eval()
    
    image_scores,image_labels = [],[]
    # image_score is highest squared difference among pixels
    # image_label is actual 0/1 encoding for good/defective
    
    pixel_scores,pixel_labels = [],[]
    # pixel_score is pixel by pixel squared difference flatten
    # pixel_label is pixel by pixel mask image flattened (0 for good part, 1 for defective portion)

    with torch.no_grad():
        for images,labels,masks in test_loader:
            images = images.to(device)
            features = extractor(images)
            reconstr = autoencoder(features)
            
            mse_map = torch.mean((features-reconstr)**2,dim=1,keepdim=True)
            
            # Ground_Truth is 256x256 but mse_map is only 32x32
            anomaly_map = nn.functional.interpolate(mse_map,size=(img_size,img_size),mode="bilinear",align_corners=False)
            anomaly_map = anomaly_map.cpu().squeeze().numpy()
            mask = masks.squeeze().numpy()
            
            image_scores.append(np.max(anomaly_map))
            image_labels.append(labels.item())
            
            pixel_scores.extend(anomaly_map.flatten())
            pixel_labels.extend((mask.flatten()>0).astype(int))
            
    img_auroc = roc_auc_score(image_labels,image_scores)
    pxl_auroc = roc_auc_score(pixel_labels,pixel_scores)
    
    print(f"[{category.upper()}] Image AUROC : {img_auroc:.4f}  |  Pixel AUROC : {pxl_auroc:.4f}")        

**Interpretation of Evaluation**

mse_map is the mean squared error across the channel depth between features and reconstructed features (reconstr)

Ground truth is 256x256 while mse_map is only 32x32, so we interpolate back to 256x256 bilinearly (anomaly_map)

image_score is the highest valued "pixel"/number in this anomaly_map
image_label is the 0/1 label for good/defective object

pixel_score is the flattened anomaly_map

pixel_label is the the flatten mask image (after scaling pixels to 0-1) : So defective portion's pixels are 1 and normal pixels are 0

## Performance

In [17]:
if __name__ == "__main__":
    target_categories = ["bottle","cable","capsule","carpet","grid","hazelnut","leather","metal_nut","pill","screw","tile","toothbrush","transistor","wood","zipper"]
    
    for cat in target_categories:
        print("\n" + "-"*50)
        print(f"Starting Pipeline for : {cat.capitalize()}")
        print("-"*50)
        
        train_evaluate(category=cat,root_dir="./datasets/MVTecAD",epochs=20)
        print()



--------------------------------------------------
Starting Pipeline for : Bottle
--------------------------------------------------
Autoencoder training for Bottle
Epoch 1/20  |  MSE Loss : 0.0938
Epoch 2/20  |  MSE Loss : 0.0271
Epoch 3/20  |  MSE Loss : 0.0148
Epoch 4/20  |  MSE Loss : 0.0102
Epoch 5/20  |  MSE Loss : 0.0082
Epoch 6/20  |  MSE Loss : 0.0070
Epoch 7/20  |  MSE Loss : 0.0064
Epoch 8/20  |  MSE Loss : 0.0059
Epoch 9/20  |  MSE Loss : 0.0056
Epoch 10/20  |  MSE Loss : 0.0053
Epoch 11/20  |  MSE Loss : 0.0051
Epoch 12/20  |  MSE Loss : 0.0049
Epoch 13/20  |  MSE Loss : 0.0048
Epoch 14/20  |  MSE Loss : 0.0046
Epoch 15/20  |  MSE Loss : 0.0045
Epoch 16/20  |  MSE Loss : 0.0043
Epoch 17/20  |  MSE Loss : 0.0042
Epoch 18/20  |  MSE Loss : 0.0041
Epoch 19/20  |  MSE Loss : 0.0040
Epoch 20/20  |  MSE Loss : 0.0039
Evaluating Bottle
[BOTTLE] Image AUROC : 0.9857  |  Pixel AUROC : 0.9679


--------------------------------------------------
Starting Pipeline for : Cable
-------

# Analysis

### Excellent performance on categories like Bottle,Hazelnut,Leather
Bottle,Hazelnut : Shot in same position,scale,orientation every time and the ResNet's receptive field captures the exact same context for every patch and the autoencoder learns a very strict definition of bottle edge or hazelnut tip, etc 

Leather : It is a repeating texture with patches looking similar across all positions, the autoencoder only needs to learn how "smooth" leather looks which can then be generalised across all patches of image for reconstruction

### Okayish performance on categories like Carpet,Wood
1x1 autoencoder can memorise most of the local textures of capet,wood such as threading/organic patterns but sometimes the input may be feature vector for a rare but normal wood which the autoencoder hasn't seen enough number of times to reconstruct perfectly - spiking the false positivers

### Poor performance on categories like Cable,Capsule,Transistor
Cable : Wires bend differently across different images and autoencoder treats unseen normal curves as huge defects

Capsule : Rotation and different view points make it harder for the encoder to learn and generalise features

Transistor : These contain very minute details and even slightest of variations alter resnet feature vectors drastically 

### Why is pixel_auroc better than image_auroc in most cases ?

np.max() can result in just one mismatching pixel leading to flagging as defect making it very sensitive to false positives.
